# Orvix — Kaggle pilot: log anomaly detection on BGL

**Purpose:** train and evaluate a small, real model for Orvix's *first stage*: flagging anomalous log messages. This notebook fine-tunes a pretrained DistilBERT encoder on a small, time-ordered sample of the labeled BGL supercomputer log dataset.

**Important scope boundary:** this is an anomaly-detection pilot, **not** a complete root-cause-localization model. BGL labels log entries as normal/alert; they do not label a microservice root cause. RCA localization is a separate stage using RCAEval's labeled microservice failure cases.

## What this pilot is designed to prove

1. The team can acquire and preprocess a public labeled log dataset.
2. The team can fine-tune model weights—not merely call an LLM API.
3. Results are evaluated on a held-out, later portion of the sampled stream.
4. A data-scaling ablation compares training on fewer versus more examples while keeping validation and test sets fixed.
5. The saved model, test metrics, and preprocessing steps form an auditable baseline for the lab-scale experiment.

The pilot intentionally samples up to 30,000 lines so it is manageable on a hosted notebook. The complete BGL corpus has about 4.75 million messages; Thunderbird contains about 211 million messages (~29.6 GiB raw) in the Loghub catalog. Dataset size alone does **not** prove why a model performs poorly; use the measured scaling results to support the next experiment.

### Before running

- In Kaggle Notebook settings, enable **Internet** (for the dataset and pretrained encoder) and select an available **GPU**.
- If the direct download is blocked by your network, download the official BGL archive from Loghub/Zenodo and attach it as a Kaggle dataset, then adapt the path in the download cell.
- The code makes no claim about expected accuracy. Your executed notebook outputs are the results to show your coordinator.

## 1. Install/import dependencies

In [ ]:
!pip -q install -U transformers scikit-learn accelerate

In [ ]:
import os, re, json, math, time, zipfile, urllib.request, gc, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Download the official BGL log archive

BGL is a supercomputer system log dataset. In the raw format, the first token is `-` for a normal message; other first-token values represent alert categories. This notebook collapses all non-`-` categories into a single anomaly label.

In [ ]:
DATA_DIR = Path("/kaggle/working/orvix_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "BGL.zip"
EXTRACT_DIR = DATA_DIR / "BGL"

BGL_URL = "https://zenodo.org/records/8196385/files/BGL.zip?download=1"

if not any(EXTRACT_DIR.rglob("*.log")) and not any(EXTRACT_DIR.rglob("*.txt")):
    if not ZIP_PATH.exists():
        print("Downloading BGL archive...")
        urllib.request.urlretrieve(BGL_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

candidates = list(EXTRACT_DIR.rglob("*.log")) + list(EXTRACT_DIR.rglob("*.txt"))
if not candidates:
    # Fall back to any sufficiently large non-archive file in the extracted folder.
    candidates = [p for p in EXTRACT_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in {".zip", ".md5"}]
if not candidates:
    raise FileNotFoundError(
        f"No log file found after extracting {ZIP_PATH}. Attach the official BGL archive as a Kaggle Dataset and update EXTRACT_DIR."
    )
LOG_PATH = max(candidates, key=lambda p: p.stat().st_size)
print("Log file:", LOG_PATH)
print("Extracted file size (MiB):", round(LOG_PATH.stat().st_size / (1024**2), 2))

## 3. Build a manageable sample and labels

To avoid using only the beginning of the stream, the notebook first counts lines and then samples approximately evenly across the file, preserving chronological order. It also removes common timestamp/node fields from the text used by the classifier to reduce trivial timestamp-based shortcuts. The label remains the official normal-versus-alert label.

In [ ]:
MAX_SAMPLED_ROWS = 30_000  # Raise only after the first run succeeds.

# Pass 1: count lines so we can sample throughout the stream.
with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
    total_lines = sum(1 for _ in f)

stride = max(1, total_lines // MAX_SAMPLED_ROWS)
rows = []
with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i % stride != 0:
            continue
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        label_token = parts[0]
        label = 0 if label_token == "-" else 1

        # BGL commonly stores: label, Unix timestamp, date, node, timestamp, then log fields.
        # Omit the label/time/node metadata and classify the remaining system message.
        message = " ".join(parts[5:]) if len(parts) > 5 else " ".join(parts[1:])
        message = re.sub(r"\s+", " ", message).strip()
        if message:
            rows.append({"text": message[:2000], "label": int(label), "source_line": i})
        if len(rows) >= MAX_SAMPLED_ROWS:
            break

df = pd.DataFrame(rows)
if len(df) < 1000 or df["label"].nunique() < 2:
    raise ValueError(
        f"Only {len(df)} usable rows or fewer than two classes were found. Inspect the source file format and label field."
    )

print("Raw lines:", total_lines)
print("Sampled usable rows:", len(df))
print("Label counts (0=normal, 1=anomaly):")
print(df["label"].value_counts().sort_index())
print("Anomaly prevalence:", round(df["label"].mean() * 100, 2), "%")
display(df.head())

## 4. Split by stream order to reduce leakage

The earliest 70% of sampled rows are training data, the next 15% are validation data, and the final 15% are test data. Validation is used to select a classification threshold; the test set is used only for final reporting. This is stricter than a random row split because nearby log messages can be highly correlated.

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = df.iloc[:train_end].reset_index(drop=True)
val_df = df.iloc[train_end:val_end].reset_index(drop=True)
test_df = df.iloc[val_end:].reset_index(drop=True)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, "rows=", len(part), "class counts=", part.label.value_counts().to_dict())
    if part.label.nunique() < 2:
        raise ValueError(f"The {name} split has only one label; increase MAX_SAMPLED_ROWS or inspect the sampling.")

## 5. Tokenize logs

This pilot uses `distilbert-base-uncased` as a compact, pretrained Transformer encoder and fine-tunes all its task-relevant weights for binary log classification. This is transfer learning, not pretraining a foundation model from scratch. If the course requirement explicitly demands training from random initialization, ask the coordinator whether fine-tuning counts; otherwise add a small randomly initialized Transformer experiment as a separate baseline.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 96
BATCH_SIZE = 16

print("Loading tokenizer and base model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LogTextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=96):
        enc = tokenizer(
            frame["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(frame["label"].astype(int).to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_ds = LogTextDataset(train_df, tokenizer, MAX_LENGTH)
val_ds = LogTextDataset(val_df, tokenizer, MAX_LENGTH)
test_ds = LogTextDataset(test_df, tokenizer, MAX_LENGTH)
print("Tokenized tensors:", train_ds.input_ids.shape, val_ds.input_ids.shape, test_ds.input_ids.shape)

## 6. Train three data-size variants

This small scaling ablation trains the same model and evaluates it on the exact same validation/test windows, while changing only the number of training examples. It helps answer whether more in-domain examples improve the pilot. Results may be noisy, and low accuracy does not automatically mean the dataset is too small; logs are imbalanced and difficult, and the label/task setup may also be responsible.

In [ ]:
def get_probabilities(model, dataset, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    probs_all = []
    labels_all = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            probs_all.extend(probs.detach().cpu().numpy().tolist())
            labels_all.extend(batch["labels"].numpy().tolist())
    return np.asarray(probs_all), np.asarray(labels_all)


def choose_threshold(y_true, probs):
    # Select threshold on validation only; never select it from the test set.
    best = (0.5, -1.0)
    for threshold in np.linspace(0.05, 0.95, 91):
        preds = (probs >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best[1]:
            best = (float(threshold), float(score))
    return best


def summarize_metrics(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision_anomaly": float(precision_score(y_true, preds, zero_division=0)),
        "recall_anomaly": float(recall_score(y_true, preds, zero_division=0)),
        "f1_anomaly": float(f1_score(y_true, preds, zero_division=0)),
        "pr_auc_average_precision": float(average_precision_score(y_true, probs)),
        "confusion_matrix_tn_fp_fn_tp": confusion_matrix(y_true, preds, labels=[0, 1]).ravel().tolist(),
    }


def train_one_model(train_subset, run_name, epochs=1):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.to(device)

    train_subset_ds = LogTextDataset(train_subset, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_subset_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    counts = np.bincount(train_subset["label"].astype(int).to_numpy(), minlength=2)
    if min(counts) == 0:
        raise ValueError(f"Training subset {run_name} does not contain both classes: {counts.tolist()}")
    class_weights = len(train_subset) / (2.0 * counts.astype(np.float32))
    class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

    history = []
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for step, batch in enumerate(train_loader, start=1):
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = torch.nn.functional.cross_entropy(logits, labels, weight=class_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += float(loss.item())
            if step % 100 == 0 or step == len(train_loader):
                print(f"{run_name} | epoch {epoch+1}/{epochs} | step {step}/{len(train_loader)} | avg loss {running_loss/step:.4f}")
        history.append({"epoch": epoch + 1, "mean_train_loss": running_loss / max(1, len(train_loader))})

    val_probs, val_y = get_probabilities(model, val_ds)
    threshold, val_f1 = choose_threshold(val_y, val_probs)
    test_probs, test_y = get_probabilities(model, test_ds)
    metrics = summarize_metrics(test_y, test_probs, threshold)
    metrics.update({
        "run": run_name,
        "n_train": int(len(train_subset)),
        "val_f1_at_selected_threshold": float(val_f1),
        "train_seconds": round(time.time() - start, 2),
        "train_loss_history": history,
    })

    if len(train_subset) == int(len(train_df)):
        out_dir = Path("/kaggle/working/orvix_bgl_pilot_model")
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        with open(out_dir / "pilot_config.json", "w") as f:
            json.dump({
                "dataset": "Loghub BGL",
                "task": "line-level normal-vs-anomaly classification",
                "base_model": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "threshold_from_validation": threshold,
                "label_mapping": {"0": "normal", "1": "anomalous/alert"},
                "note": "This is not a root-cause-localization model.",
            }, f, indent=2)
        pd.DataFrame({"text": test_df["text"], "label": test_y, "anomaly_probability": test_probs,
                      "prediction": (test_probs >= threshold).astype(int)}).to_csv(
            "/kaggle/working/orvix_bgl_pilot_test_predictions.csv", index=False)
        print("Saved final pilot model to", out_dir)

    del model, train_subset_ds, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics

# Compare small / medium / full available pilot training sizes.
# The largest run uses the full 70% training split; validation and test never change.
train_sizes = sorted(set([min(3_000, len(train_df)), min(10_000, len(train_df)), len(train_df)]))
results = []
for size in train_sizes:
    if size == len(train_df):
        subset = train_df.reset_index(drop=True)
    else:
        # Evenly sample across the full training period instead of taking only its beginning.
        selected = np.linspace(0, len(train_df) - 1, num=size, dtype=int)
        subset = train_df.iloc[selected].reset_index(drop=True)
    if subset.label.nunique() < 2:
        print(f"Skipping size {size}: sampled subset has only one class; increase MAX_SAMPLED_ROWS")
        continue
    result = train_one_model(subset, run_name=f"train_{size}", epochs=1)
    results.append(result)
    print("Finished:", result)

results_df = pd.DataFrame([{k:v for k,v in row.items() if k != "train_loss_history"} for row in results])
results_df.to_csv("/kaggle/working/orvix_pilot_scaling_results.csv", index=False)
display(results_df)

## 7. Interpret the metrics correctly

- **Accuracy is insufficient** because most lines are normal. A model predicting every line as normal can have deceptively high accuracy.
- **Anomaly precision** tells you what fraction of alerts were correct.
- **Anomaly recall** tells you what fraction of labeled anomalous lines were found.
- **Anomaly F1** balances precision and recall at the selected threshold.
- **Average precision / PR-AUC** is helpful under class imbalance.
- **Confusion matrix `[TN, FP, FN, TP]`** shows the trade-off between false alarms and missed anomalies.
- The three training-size runs are a data-scaling ablation. Do not promise that the largest run must win; report what actually happens and investigate if it does not.

This benchmark evaluates BGL line-level anomaly detection. It does **not** prove that the model can identify a root-cause service in a microservice incident.

## 8. Export the evidence for the lab coordinator

After the notebook finishes, download these files from the Kaggle output panel:

- `orvix_bgl_pilot_model/` — fine-tuned model weights, tokenizer, and config.
- `orvix_pilot_scaling_results.csv` — metrics versus training-set size.
- `orvix_bgl_pilot_test_predictions.csv` — held-out test predictions for error analysis.

Also save the notebook with its outputs as HTML or PDF. Keep a copy of the exact dataset version, notebook, hyperparameters, package versions, and random seed for reproducibility.

## 9. Next stage: RCA localization (not included in this notebook)

Use [RCAEval](https://github.com/phamquiluan/RCAEval) for supervised root-cause localization. It provides 735 labeled microservice failure cases across Online Boutique, Sock Shop, and Train Ticket, including root-cause service annotations; RE2/RE3 include logs (and some suites include traces). Start with a small subset for pipeline validation, then evaluate on a held-out incident split. Do not mix HDFS/BGL line-level anomaly labels with RCAEval service-level root-cause labels as if they were the same task.

The lab-scale phase should then test a reproducible scale-up on large log corpora (e.g., HDFS/BGL and, with appropriate storage/compute, Thunderbird) and evaluate whether the model generalizes to new systems. More raw logs help representation learning and scale testing, but root-cause labels and cross-system evaluation remain necessary.